# Assignment 2 — Sentiment Analysis and Topic Modelling

## Part A — Sentiment Analysis

### Q1. Load and Preprocess

Load sentiment_sentences.csv using pandas. Preprocess the text column — convert to lower case, remove punctuation and special characters, tokenize, and remove stop words. Print the cleaned tokens for one sample review.

In [1]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure required NLTK data is available (first-run will download)
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1028)>


False

In [2]:
# Read the CSV into a DataFrame
data = pd.read_csv('sentiment_sentences.csv')
print('Rows, columns:', data.shape)
print('\nColumns:', data.columns.tolist())
print('\nSample:')
print(data.head())

Rows, columns: (3000, 3)

Columns: ['text', 'sentiment', 'source']

Sample:
                                                text  sentiment  source
0  So there is no way for me to plug it in here i...          0  amazon
1                        Good case, Excellent value.          1  amazon
2                             Great for the jawbone.          1  amazon
3  Tied to charger for conversations lasting more...          0  amazon
4                                  The mic is great.          1  amazon


In [3]:
# Prepare English stopwords set
eng_stop = set(stopwords.words('english'))

def clean_sentence(sent):
    """Lowercase, strip non-letters, tokenize and drop stopwords."""
    if not isinstance(sent, str):
        return []
    s = sent.lower()
    s = re.sub(r'[^a-z\s]', ' ', s)  # keep letters and spaces only
    toks = word_tokenize(s)
    toks = [t for t in toks if t and t not in eng_stop]
    return toks

# Apply cleaning and store tokens in a new column
data['cleaned_tokens'] = data['text'].apply(clean_sentence)

# Show cleaned tokens for the first example
print('Original:', data.loc[0, 'text'])
print('Cleaned tokens:', data.loc[0, 'cleaned_tokens'])

Original: So there is no way for me to plug it in here in the US unless I go by a converter.
Cleaned tokens: ['way', 'plug', 'us', 'unless', 'go', 'converter']


### Q2. Naive Bayes from scratch + Evaluation

Split the data into training (80%) and test (20%) sets using a stratified split. Implement the Multinomial Naive Bayes classifier from scratch — compute the class priors P(c) = N_c / N and the word likelihoods with Laplace (add-1) smoothing: P(w|c) = (count(w,c) + 1) / (sum of counts in c + |V|). Predict the sentiment of the test set and report the Accuracy and the F1-score.

In [4]:
# Stratified split keeping label balance
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data['cleaned_tokens'].tolist(),
    data['sentiment'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=data['sentiment']
)

print('Train size:', len(train_texts))
print('Test size:', len(test_texts))
print('Train distribution:', Counter(train_labels))
print('Test distribution:', Counter(test_labels))

Train size: 2400
Test size: 600
Train distribution: Counter({1: 1200, 0: 1200})
Test distribution: Counter({1: 300, 0: 300})


In [5]:
class SimpleMultinomialNB:
    """A concise Multinomial Naive Bayes with Laplace smoothing."""
    def __init__(self):
        self.prior = {}
        self.likelihood = {}
        self.vocab = []
        self.labels = []

    def fit(self, docs, labels):
        # labels: list of class labels, docs: list of token lists
        self.labels = sorted(set(labels))
        n = len(labels)
        # build vocabulary
        vocab_set = set()
        for tokens in docs:
            vocab_set.update(tokens)
        self.vocab = sorted(vocab_set)
        V = len(self.vocab)
        print('Vocab size:', V)
        # priors
        for lab in self.labels:
            self.prior[lab] = labels.count(lab) / n
        print('Priors:', self.prior)
        # likelihoods with add-1 smoothing
        for lab in self.labels:
            # collect tokens from docs with this label
            token_counter = Counter()
            total = 0
            for i, l in enumerate(labels):
                if l == lab:
                    token_counter.update(docs[i])
                    total += len(docs[i])
            self.likelihood[lab] = {}
            for w in self.vocab:
                cnt = token_counter.get(w, 0)
                self.likelihood[lab][w] = (cnt + 1) / (total + V)

    def predict(self, docs):
        preds = []
        for tokens in docs:
            best_lab = None
            best_score = -1e12
            for lab in self.labels:
                score = np.log(self.prior[lab])
                for t in tokens:
                    if t in self.likelihood[lab]:
                        score += np.log(self.likelihood[lab][t])
                if score > best_score:
                    best_score = score
                    best_lab = lab
            preds.append(best_lab)
        return preds

In [6]:
# Train and evaluate
clf = SimpleMultinomialNB()
clf.fit(train_texts, train_labels)
preds = clf.predict(test_texts)
acc = accuracy_score(test_labels, preds)
f1 = f1_score(test_labels, preds)
print(f'Accuracy: {acc:.4f}')
print(f'F1-Score: {f1:.4f}')

Vocab size: 4351
Priors: {0: 0.5, 1: 0.5}
Accuracy: 0.8117
F1-Score: 0.8132
Accuracy: 0.8117
F1-Score: 0.8132


## Part B — Topic Modelling with LDA

### Q3. Document-Term Matrix, LDA and Topic Words

Build the document-term (word-count) matrix from the review sentences after removing English stop words (you may use CountVectorizer). Apply the LDA algorithm to create K = 3 topics, then list the top 10 highest-probability words in each topic.

In [7]:
# Create document-term matrix, auto-dropping English stopwords
vec = CountVectorizer(stop_words='english')
dt_matrix = vec.fit_transform(data['text'])
print('DTM shape:', dt_matrix.shape)
print('Vocabulary size:', len(vec.vocabulary_))

DTM shape: (3000, 4896)
Vocabulary size: 4896


In [8]:
# Fit LDA with three topics
model_lda = LatentDirichletAllocation(n_components=3, random_state=42, max_iter=10)
model_lda.fit(dt_matrix)
print('LDA trained (3 topics)')

LDA trained (3 topics)


In [9]:
# helper to show top words per topic
feature_names = vec.get_feature_names_out()
def show_top_words(lda_model, features, n_top=10):
    for t_idx, comp in enumerate(lda_model.components_):
        top_idx = comp.argsort()[:-n_top-1:-1]
        words = [features[i] for i in top_idx]
        print(f'Topic #{t_idx}:', ' '.join(words))
    print()

print('Top 10 words per topic:')
show_top_words(model_lda, feature_names, n_top=10)

Top 10 words per topic:
Topic #0: phone film just like battery love does disappointed ve terrible
Topic #1: great good food service place bad just quality time works
Topic #2: movie film don time really recommend place good sound best



### Q4. Document-Topic Distribution

For at least 5 documents, print the distribution over the 3 topics and the dominant (most probable) topic. Optionally, compare the dominant topic against the source column to see how the discovered topics relate to the three review domains.

In [10]:
# Get per-document topic probabilities
doc_topics = model_lda.transform(dt_matrix)
print('Doc-topic shape:', doc_topics.shape)

Doc-topic shape: (3000, 3)


In [11]:
# Print distribution and dominant topic for five examples
for i in range(5):
    print('\nDocument #', i)
    print('Text:', data.loc[i, 'text'])
    print('Source:', data.loc[i, 'source'])
    print('Sentiment:', data.loc[i, 'sentiment'])
    probs = doc_topics[i]
    print('Topic distribution:', probs)
    print('Dominant topic: Topic #', int(np.argmax(probs)))


Document # 0
Text: So there is no way for me to plug it in here in the US unless I go by a converter.
Source: amazon
Sentiment: 0
Topic distribution: [0.0684345  0.84517423 0.08639128]
Dominant topic: Topic # 1

Document # 1
Text: Good case, Excellent value.
Source: amazon
Sentiment: 1
Topic distribution: [0.07425894 0.85628089 0.06946017]
Dominant topic: Topic # 1

Document # 2
Text: Great for the jawbone.
Source: amazon
Sentiment: 1
Topic distribution: [0.11127743 0.7768542  0.11186836]
Dominant topic: Topic # 1

Document # 3
Text: Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!
Source: amazon
Sentiment: 0
Topic distribution: [0.68792613 0.15635562 0.15571824]
Dominant topic: Topic # 0

Document # 4
Text: The mic is great.
Source: amazon
Sentiment: 1
Topic distribution: [0.11140114 0.76590538 0.12269348]
Dominant topic: Topic # 1


In [12]:
# Add dominant topic to DataFrame and cross-tabulate with source
data['dominant_topic'] = np.argmax(doc_topics, axis=1)
print('Cross-tab of source vs dominant topic:')
print(pd.crosstab(data['source'], data['dominant_topic']))

Cross-tab of source vs dominant topic:
dominant_topic    0    1    2
source                       
amazon          323  373  304
imdb            289  248  463
yelp            230  498  272


In [13]:
# Topic distribution percentages by source
print('Topic distribution by source:')
for src in data['source'].unique():
    subset = data[data['source'] == src]
    pct = subset['dominant_topic'].value_counts(normalize=True).sort_index()
    print(f'\n{src}:')
    for topic, p in pct.items():
        print(f'  Topic {int(topic)}: {p:.2%}')

Topic distribution by source:

amazon:
  Topic 0: 32.30%
  Topic 1: 37.30%
  Topic 2: 30.40%

yelp:
  Topic 0: 23.00%
  Topic 1: 49.80%
  Topic 2: 27.20%

imdb:
  Topic 0: 28.90%
  Topic 1: 24.80%
  Topic 2: 46.30%
